# Experiment — Does `class_weight='balanced'` Improve Recall on Netflix Content Moderation?

Dựa trên kiến trúc thử nghiệm chuẩn của **Section 8**, notebook này áp dụng bài toán phân loại và kiểm soát nội dung cho tập dữ liệu **`netflix_titles_cleaned.csv`**:
- **Bài toán nghiệp vụ**: Tự động phát hiện và kiểm duyệt phim dành cho người lớn (**Adults 18+**: `TV-MA`, `R`, `NC-17`).
- **Mục tiêu thử nghiệm**: So sánh mô hình **Baseline** (`class_weight=None`) và mô hình **Balanced** (`class_weight='balanced'`) cùng kỹ thuật **Threshold Tuning** để trả lời câu hỏi: *Làm thế nào để tối đa hóa độ nhạy (**Recall**) nhằm ngăn chặn triệt để phim 18+ bị lọt vào hồ sơ của trẻ em?*

## Step 1 — Load the Data, Pick Features and Target

- Nạp dữ liệu phim từ file `netflix_titles_cleaned.csv`.
- **Target ($y$)**: Nhãn nhị phân `is_adult` (1 nếu là phim 18+ gồm `TV-MA`, `R`, `NC-17`; 0 nếu là phim dành cho gia đình & thanh thiếu niên).
- **Features ($X$)**: Tổ hợp các thuộc tính văn bản giàu ngữ cảnh: `type` (Phim lẻ/Bộ), `listed_in` (Thể loại) và `description` (Tóm tắt nội dung).

In [1]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu Netflix
df = pd.read_csv('netflix_titles_cleaned.csv')

# 2. Xác định nhãn mục tiêu y (Adult 18+ vs Non-Adult)
adult_ratings = ['TV-MA', 'R', 'NC-17']
df['is_adult'] = df['rating'].isin(adult_ratings).astype(int)

# 3. Ghép đặc trưng văn bản X
X = df['type'] + ' | ' + df['listed_in'] + ' | ' + df['description']
y = df['is_adult']

print(f"Tổng số mẫu dữ liệu: {len(df)}")
print("\nPhân bố nhãn mục tiêu (is_adult):")
print(y.value_counts(normalize=True).mul(100).round(2).astype(str) + '% (0: Non-Adult, 1: Adult 18+)')

df[['title', 'rating', 'is_adult', 'listed_in']].head()

Tổng số mẫu dữ liệu: 8807

Phân bố nhãn mục tiêu (is_adult):
is_adult
0    54.48% (0: Non-Adult, 1: Adult 18+)
1    45.52% (0: Non-Adult, 1: Adult 18+)
Name: proportion, dtype: str


,title,rating,is_adult,listed_in
0,Dick Johnson Is Dead,PG-13,0,Documentaries
1,Blood & Water,TV-MA,1,"International TV Shows, TV Dramas, TV Mysteries"
2,Ganglands,TV-MA,1,"Crime TV Shows, International TV Shows, TV Act..."
3,Jailbirds New Orleans,TV-MA,1,"Docuseries, Reality TV"
4,Kota Factory,TV-MA,1,"International TV Shows, Romantic TV Shows, TV ..."


## Step 2 — Encode Text Columns, Then Split

- Khác với dữ liệu bảng dạng số dùng `pd.get_dummies()`, với dữ liệu văn bản phim ảnh, ta mã hóa đặc trưng bằng **`TfidfVectorizer`** (chuyển đổi văn bản thành ma trận trọng số tần suất từ vựng TF-IDF).
- Tách tập dữ liệu thành **80% Train** và **20% Test** độc lập bằng `train_test_split()` với `stratify=y`.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Phân chia Train/Test trước để chống rò rỉ dữ liệu (data leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Vector hóa văn bản bằng TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=8000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test : {X_test.shape}")

Kích thước tập Train: (7045, 8000)
Kích thước tập Test : (1762, 8000)


## Step 3 — Train the Logistic Regression WITH `class_weight='balanced'`

### What is `class_weight`?

Đó là thiết lập điều hướng thuật toán: **"coi sai sót trên nhóm nhãn này quan trọng hơn (hoặc ít quan trọng hơn) sai sót trên nhóm nhãn khác."**

**Tại sao cần thiết ở đây:**
Thông thường, mô hình chỉ cố gắng tối thiểu hóa *tổng số lỗi* mà không quan tâm lỗi rơi vào lớp nào. Nhưng trong nghiệp vụ kiểm duyệt của Netflix:
- Bỏ lọt 1 bộ phim 18+ (gán nhãn 0 cho phim bạo lực/người lớn) khiến trẻ em vô tình xem phải nội dung độc hại là **thảm họa nghiêm trọng** về uy tín và pháp lý.
- `class_weight='balanced'` giải quyết điều này bằng cách phạt nặng hơn khi mô hình đoán sai nhóm quan trọng, buộc ranh giới quyết định dịch chuyển để bắt trọn nhiều phim 18+ hơn.

### Ví dụ 1 — Tương tự như chấm thi có câu hỏi điểm liệt
Hãy tưởng tượng một bài thi có những câu hỏi mang hệ số điểm gấp 3 lần câu thường. Bạn sẽ dành sự chú ý đặc biệt để không làm sai câu đó. `class_weight='balanced'` chính là quy tắc chấm điểm đó trong Machine Learning.

### Ví dụ 2 — Trường hợp một bộ phim ranh giới (Borderline Movie)
Xét một bộ phim trinh thám u tối mà mô hình dự đoán độ tin cậy 18+ là 45% (ngay dưới ngưỡng 50% mặc định):
- **Không có `class_weight`**: Mô hình đoán là "Không phải 18+" vì đó là phương án an toàn giảm lỗi tổng quát.
- **Có `class_weight='balanced'`**: Phạt nặng lỗi bỏ sót phim 18+, mô hình chủ động cảnh báo phim này là 18+ để nhân viên kiểm duyệt kiểm tra lại.

### Ví dụ 3 — Trọng số được tính toán như thế nào?
Công thức của Scikit-Learn: `n_samples / (n_classes * count_of_class)`.
- Lớp 0 (Non-Adult): $7045 / (2 \times 3838) \approx 0.92$
- Lớp 1 (Adult 18+): $7045 / (2 \times 3207) \approx 1.10$

In [3]:
from sklearn.linear_model import LogisticRegression

# Huấn luyện mô hình Logistic Regression với class_weight='balanced'
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Đồng thời huấn luyện thêm mô hình Baseline (không dùng class_weight) để đối chiếu
baseline_model = LogisticRegression(max_iter=1000, class_weight=None, random_state=42)
baseline_model.fit(X_train, y_train)

print("✓ Đã huấn luyện xong cả mô hình Baseline và Balanced!")

✓ Đã huấn luyện xong cả mô hình Baseline và Balanced!


## Step 4 — Predict on the Test Set

Thực hiện dự đoán nhãn trên tập kiểm tra (`X_test`) và tính điểm Accuracy tổng quan.

In [4]:
# Dự đoán từ mô hình Balanced
y_pred = model.predict(X_test)
acc_balanced = model.score(X_test, y_test)

print(f"Mean Accuracy trên tập Test (Balanced model): {acc_balanced:.4f} ({acc_balanced*100:.2f}%)")

Mean Accuracy trên tập Test (Balanced model): 0.7111 (71.11%)


## Step 5 — Evaluate With the Confusion Matrix

Đánh giá chi tiết bằng Ma trận nhầm lẫn (Confusion Matrix), Recall, Precision và Báo cáo phân loại (Classification Report).

In [5]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, classification_report

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=['Actual Non-Adult (0)', 'Actual Adult (1)'], 
                      columns=['Pred Non-Adult (0)', 'Pred Adult (1)']))

print("\nRecall (Adults 18+)   :", round(recall_score(y_test, y_pred), 4))
print("Precision (Adults 18+):", round(precision_score(y_test, y_pred), 4))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Non-Adult (0)', 'Adult 18+ (1)'], digits=4))

Confusion Matrix:
                      Pred Non-Adult (0)  Pred Adult (1)
Actual Non-Adult (0)                 681             279
Actual Adult (1)                     230             572

Recall (Adults 18+)   : 0.7132
Precision (Adults 18+): 0.6722

--- Classification Report ---
               precision    recall  f1-score   support

Non-Adult (0)     0.7475    0.7094    0.7280       960
Adult 18+ (1)     0.6722    0.7132    0.6921       802

     accuracy                         0.7111      1762
    macro avg     0.7098    0.7113    0.7100      1762
 weighted avg     0.7132    0.7111    0.7116      1762



## Step 6 — Baseline vs. Balanced — Side by Side

Bảng so sánh trực diện giữa mô hình cơ sở (**Baseline**) và mô hình cân bằng (**Balanced**):

In [6]:
# Tính toán các chỉ số cho Baseline model
y_pred_base = baseline_model.predict(X_test)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, y_pred_base).ravel()
tn_bal, fp_bal, fn_bal, tp_bal = confusion_matrix(y_test, y_pred).ravel()

comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Recall (Adults 18+)",
        "Precision (Adults 18+)",
        "Phim 18+ bị bỏ lọt (False Negatives - Báo thiếu)",
        "Báo động nhầm (False Positives - Báo thừa)"
    ],
    "Baseline (class_weight=None)": [
        f"{baseline_model.score(X_test, y_test)*100:.1f}%",
        f"{recall_score(y_test, y_pred_base)*100:.1f}%",
        f"{precision_score(y_test, y_pred_base)*100:.1f}%",
        fn_b,
        fp_b
    ],
    "Balanced (class_weight='balanced')": [
        f"{model.score(X_test, y_test)*100:.1f}%",
        f"{recall_score(y_test, y_pred)*100:.1f}%",
        f"{precision_score(y_test, y_pred)*100:.1f}%",
        fn_bal,
        fp_bal
    ]
})

display(comparison_df)

,Metric,Baseline (class_weight=None),Balanced (class_weight='balanced')
0,Accuracy,71.4%,71.1%
1,Recall (Adults 18+),63.6%,71.3%
2,Precision (Adults 18+),70.6%,67.2%
3,Phim 18+ bị bỏ lọt (False Negatives - Báo thiếu),292,230
4,Báo động nhầm (False Positives - Báo thừa),212,279


### Phân tích kết quả thực tế (What Actually Happened?)

- **Recall tăng vọt từ 63.6% lên 71.3%**: Mô hình Balanced đã bắt thêm **62 bộ phim 18+** mà mô hình Baseline bỏ sót (số phim 18+ bị bỏ lọt giảm mạnh từ **292 xuống còn 230**).
- **Cái giá phải trả**: Precision giảm nhẹ từ 70.6% xuống 67.2%, số ca báo động nhầm tăng từ 212 lên 279.

**Sự đánh đổi này có xứng đáng không?**
> **Hoàn toàn xứng đáng!** Trong nghiệp vụ kiểm duyệt bảo vệ trẻ em, một cảnh báo nhầm (False Alarm) chỉ tốn vài phút của nhân viên kiểm duyệt kiểm tra lại. Nhưng một bộ phim 18+ bị bỏ lọt (False Negative) lọt vào tài khoản trẻ em sẽ gây hậu quả pháp lý nghiêm trọng và tổn hại uy tín thương hiệu Netflix.

## Step 7 — Threshold Tuning

`class_weight='balanced'` thay đổi cách mô hình *học*. Còn **Threshold Tuning (Tinh chỉnh ngưỡng quyết định)** thay đổi cách chúng ta *đọc kết quả* — hoàn toàn không cần huấn luyện lại mô hình!

Mặc định, `.predict()` chỉ gán nhãn là "Adult 18+" khi độ tin cậy trên **50% (0.5)**. Nhưng con số 50% chỉ là mặc định kỹ thuật. Vì bỏ sót phim 18+ quá nguy hiểm, ta có thể hạ ngưỡng xuống (ví dụ: 0.4, 0.3) để giăng lưới bắt sạch phim 18+.

In [7]:
# Lấy xác suất dự đoán là Adult 18+ của từng bộ phim
probs = model.predict_proba(X_test)[:, 1]
print("10 xác suất đầu tiên của tập Test:")
print(np.round(probs[:10], 4))

10 xác suất đầu tiên của tập Test:
[0.3696 0.1975 0.5953 0.4788 0.6952 0.4928 0.3846 0.4101 0.4873 0.0594]


In [8]:
from sklearn.metrics import recall_score, precision_score, accuracy_score, confusion_matrix

thresholds = [0.6, 0.5, 0.4, 0.3, 0.2]
results = []

for t in thresholds:
    y_pred_t = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    results.append({
        "Threshold (Ngưỡng)": t,
        "Accuracy": f"{accuracy_score(y_test, y_pred_t)*100:.2f}%",
        "Recall (Adults)": f"{recall_score(y_test, y_pred_t)*100:.2f}%",
        "Precision (Adults)": f"{precision_score(y_test, y_pred_t)*100:.2f}%",
        "Missed Adult 18+ (FN)": fn,
        "False Alarms (FP)": fp
    })

tuning_df = pd.DataFrame(results)
display(tuning_df)

,Threshold (Ngưỡng),Accuracy,Recall (Adults),Precision (Adults),Missed Adult 18+ (FN),False Alarms (FP)
0,0.6,69.92%,52.49%,73.86%,381,149
1,0.5,71.11%,71.32%,67.22%,230,279
2,0.4,67.65%,86.53%,60.03%,108,462
3,0.3,62.37%,96.01%,54.96%,32,631
4,0.2,59.08%,99.25%,52.68%,6,715


### Đọc hiểu bảng Threshold Tuning

Hạ ngưỡng quyết định đồng nghĩa với: *"Cảnh báo đây là phim 18+ ngay cả khi mô hình có độ tin cậy thấp hơn."* Khi ngưỡng giảm từ **0.5 xuống 0.3**:
- **Recall tăng vọt từ 71.32% lên 96.01%**: Lưới quét cực rộng, số phim 18+ bị bỏ lọt giảm không tưởng từ **230 phim xuống chỉ còn 32 phim** trên toàn bộ 802 phim 18+ kiểm thử!
- **Precision giảm xuống 54.96%**: Tăng số lượng phim gia đình bị cảnh báo nhầm (631 phim).

### Cách chọn ngưỡng trong thực tế doanh nghiệp

Không có ngưỡng nào là "chuẩn duy nhất" — nó phụ thuộc vào năng lực xử lý của đội ngũ kiểm duyệt:
- **Nếu ưu tiên an toàn tuyệt đối cho trẻ em**: Chọn ngưỡng **$t = 0.3$** hoặc **$t = 0.4$**. Đảm bảo bắt trọn $\ge 86.5\%$ đến $96\%$ phim người lớn.
- **Nếu đội kiểm duyệt quá tải**: Chọn ngưỡng **$t = 0.5$** để cân bằng giữa khối lượng công việc và độ an toàn.